# 23N Field Tests #03 — Gujarat from Space: Night Lights as an Economic Pulse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft03-night-lights-gujarat/ft03_night_lights_gujarat.ipynb)

**Question.** Which Gujarat districts' economic activity grew fastest, *measured from space*? Night-time lights
seen by the VIIRS satellite sensor are a well-studied proxy for economic activity (electrification, industry, ports,
urban growth). We add up the light inside each of Gujarat's 33 district boundaries for every year from 2014 to the
latest available year, compute growth, and rank the districts. Then you can ask questions of the result in plain
English, and the notebook renders a 1-minute video and the post text.

**Data.**
* Night lights: *Annual global nighttime lights at 500 m, 2000–2024* (OpenGeoHub, T. Hengl, Zenodo record
  [17294744](https://zenodo.org/records/17294744), DOI [10.5281/zenodo.7750174](https://doi.org/10.5281/zenodo.7750174)), a
  Cloud-Optimized GeoTIFF repackaging of the EOG **Annual VNL V2** VIIRS product (Elvidge et al., 2021) for 2012–2024.
  Licence **CC BY 4.0**. **No login needed.**
* District boundaries: `GUJARAT_DISTRICTS.geojson` from [datta07/INDIAN-SHAPEFILES](https://github.com/datta07/INDIAN-SHAPEFILES)
  (33 districts, LGD codes). Repository licence **MIT**.

**Runtime.** About 6–15 minutes on free Colab (CPU is enough; a T4 GPU only adds the optional small-LLM answers in the demo).
Download time depends on Zenodo speed. *Independent study on public data. No affiliation with any company or government body.*

## 1 · Setup
Installs the light dependencies. `geopandas`, `matplotlib`, `pandas` and `transformers` are already on Colab and Kaggle.
On **Kaggle**, switch *Internet* on in the notebook settings first.

In [ ]:
%pip install -q rasterio geopandas gradio

import os, io, sys, json, math, time, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize
from matplotlib.ticker import FuncFormatter, NullFormatter, LogLocator
import matplotlib.patheffects as pe
STROKE = [pe.withStroke(linewidth=3, foreground="#0B0F14")]

def tidy_log_colorbar(cb, horizontal=False):
    axis = cb.ax.xaxis if horizontal else cb.ax.yaxis
    axis.set_major_locator(LogLocator(base=10, subs=(1.0, 3.0)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(NullFormatter())
warnings.filterwarnings("ignore")
T0 = time.time()

# ---- Assumptions & knobs (all named, all editable) ----------------------------------------
BASE_YEAR = 2014            # growth is measured from this year ...
LAST_YEAR_WANTED = 2024     # ... to the latest year available up to this one
VALUE_SCALE = 0.1           # OpenGeoHub stores Annual VNL V2 radiance x10 (0-2000) -> nW/cm2/sr
LIT_THRESHOLD = 1.0         # nW/cm2/sr; a pixel at or above this counts as "lit"
MIN_BASE_SHARE = 0.005      # a district needs >= 0.5% of Gujarat's base-year light to claim the headline
                            # (stops a near-dark district winning on a tiny base; the unfiltered leader is printed too)
BBOX = (68.0, 20.0, 74.6, 24.8)   # lon_min, lat_min, lon_max, lat_max around Gujarat
USE_LLM = True              # the demo uses a small open LLM if a GPU is present; otherwise a deterministic lookup
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
NOTEBOOK_URL = "https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft03-night-lights-gujarat/ft03_night_lights_gujarat.ipynb"

DATA_DIR = Path("ft03_data"); DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("ft03_figures"); FIG_DIR.mkdir(exist_ok=True)

# ---- Brand -----------------------------------------------------------------------------
BG, FG, ACCENT, TEAL, MUTED, GRID = "#0B0F14", "#F5F7FA", "#FF7A1A", "#2EC4B6", "#9AA4B2", "#1F2933"
NIGHT_CMAP = LinearSegmentedColormap.from_list(
    "night", ["#141B24", "#3A2716", "#9C4510", "#FF7A1A", "#FFC98A", "#FFF6E6"])
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": FG, "axes.labelcolor": FG, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "font.family": "DejaVu Sans", "axes.titleweight": "bold"})

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB, "| python", sys.version.split()[0])

## 2 · Data

### Why this source (lowest-friction route that works in free Colab)
| Route | Login? | Verdict |
|---|---|---|
| **OpenGeoHub annual VIIRS 500 m on Zenodo (Annual VNL V2, 2012–2024), CC BY 4.0** | **No** | **Used.** One Cloud-Optimized GeoTIFF per year. We read only the Gujarat window over HTTP, falling back to a full ~60–85 MB download per year if range reads fail. |
| Google Earth Engine (`NOAA/VIIRS/DNB/ANNUAL_V22`, `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`) | Yes | Needs a Google account, `ee.Authenticate()`, and a Cloud project registered for noncommercial Earth Engine use. Steps are below. |
| NASA Black Marble (VNP46A3/A4) via LAADS / `blackmarblepy` | Yes | Needs a NASA Earthdata login and a bearer token. |
| EOG (Colorado School of Mines) Annual VNL direct | Yes | Free EOG account required since 2022. |
| World Bank *Light Every Night* (AWS open data) | No | Raw nightly swaths (huge); you would have to build the annual composites yourself. |
| Microsoft Planetary Computer | No | We found no VIIRS night-lights collection in its catalogue. |

<details><summary>Optional: the same analysis with Google Earth Engine (exact steps)</summary>

1. Sign in at <https://code.earthengine.google.com/register>, choose **Unpaid usage → Academia & Research / Nonprofit / personal**, and create or select a Google Cloud project (for example `my-ee-project`).
2. In Colab: `import ee; ee.Authenticate(); ee.Initialize(project="my-ee-project")`.
3. `img = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22").filterDate("2014-01-01","2025-01-01").select("average")`, then
   `reduceRegions(collection=<districts FeatureCollection>, reducer=ee.Reducer.sum(), scale=500)` for each year.
This notebook does **not** need any of that.
</details>

**Assumptions:** values are rescaled with `VALUE_SCALE` (stored 0–2000 → 0–200 nW/cm²/sr). The source is a
*VNL V2 annual* composite. EOG produces it with background and ephemeral lights (fires, boats) suppressed.

In [ ]:
# ---- 2a. District boundaries (33 districts) ---------------------------------------------
BOUNDARY_SOURCES = [
    "https://raw.githubusercontent.com/datta07/INDIAN-SHAPEFILES/08490eaab1b9bb4b55addfb1d2c742586c8d22e0/STATES/GUJARAT/GUJARAT_DISTRICTS.geojson",
    "https://cdn.jsdelivr.net/gh/datta07/INDIAN-SHAPEFILES@08490eaab1b9bb4b55addfb1d2c742586c8d22e0/STATES/GUJARAT/GUJARAT_DISTRICTS.geojson",
]
import requests

def download(url, dest, tries=4, chunk=1 << 23, timeout=120):
    dest = Path(dest)
    for k in range(tries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for b in r.iter_content(chunk):
                        f.write(b)
                tmp.rename(dest)
                return dest
        except Exception as e:
            print(f"  retry {k+1}/{tries} for {url.split('/')[-1][:60]}: {e}")
            time.sleep(3 * (k + 1))
    raise RuntimeError(f"download failed: {url}")

bfile = DATA_DIR / "gujarat_districts.geojson"
if not bfile.exists():
    for u in BOUNDARY_SOURCES:
        try:
            download(u, bfile); break
        except Exception as e:
            print("boundary source failed:", e)
districts = gpd.read_file(bfile).to_crs(4326)
NAME_COL = "dtname" if "dtname" in districts.columns else [c for c in districts.columns if c != "geometry"][0]

# Common spellings -> what readers expect
PRETTY = {"Kachchh": "Kachchh (Kutch)", "Ahmadabad": "Ahmedabad", "Dohad": "Dahod", "Mahesana": "Mehsana",
          "Banas Kantha": "Banaskantha", "Sabar Kantha": "Sabarkantha", "Panch Mahals": "Panchmahal",
          "The Dangs": "Dang", "Chota Udaipur": "Chhota Udaipur"}
districts["district"] = districts[NAME_COL].astype(str).str.strip().map(lambda s: PRETTY.get(s, s))
districts = districts[["district", "geometry"]].dissolve("district").reset_index()
districts["area_km2"] = districts.to_crs(32643).area / 1e6
districts_plot = districts.copy()
districts_plot["geometry"] = districts_plot.geometry.simplify(0.004, preserve_topology=True)
print(len(districts), "districts:", ", ".join(sorted(districts.district)))

In [ ]:
# ---- 2b. Night lights: read only the Gujarat window of each annual COG -------------------
import rasterio
from rasterio.windows import from_bounds

ZENODO_RECORD = "17294744"
NTL_FILE = "nightlights.average_viirs.v21_m_500m_s_{y}0101_{y}1231_go_epsg4326_v20250904.tif"
def ntl_url(y):
    return f"https://zenodo.org/records/{ZENODO_RECORD}/files/{NTL_FILE.format(y=y)}?download=1"

GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="5", GDAL_HTTP_RETRY_DELAY="3",
                GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES", GDAL_HTTP_MULTIRANGE="YES", VSI_CACHE="TRUE",
                GDAL_HTTP_TIMEOUT="120")

def _read_window(src):
    win = from_bounds(*BBOX, transform=src.transform).round_offsets().round_lengths()
    arr = src.read(1, window=win, boundless=False).astype("float32")
    if src.nodata is not None:
        arr[arr == src.nodata] = np.nan
    return arr, src.window_transform(win)

def fetch_year(y):
    url = ntl_url(y)
    try:  # fast path: HTTP range reads of just the Gujarat tiles (a few MB)
        with rasterio.Env(**GDAL_ENV), rasterio.open("/vsicurl/" + url) as src:
            return _read_window(src) + ("range-read",)
    except Exception as e:
        print(f"  {y}: range read failed ({str(e)[:80]}); downloading full file ...")
    tmp = download(url, DATA_DIR / f"ntl_{y}.tif")
    try:
        with rasterio.open(tmp) as src:
            return _read_window(src) + ("full-download",)
    finally:
        tmp.unlink(missing_ok=True)

cache = DATA_DIR / f"gujarat_ntl_{BASE_YEAR}_{LAST_YEAR_WANTED}.npz"
NTL, NTL_TRANSFORM = {}, None
if cache.exists():
    z = np.load(cache)
    NTL = {int(k[1:]): z[k] for k in z.files if k.startswith("y")}
    NTL_TRANSFORM = rasterio.Affine(*z["transform"][:6])
    print("loaded cache", cache)
else:
    for y in range(BASE_YEAR, LAST_YEAR_WANTED + 1):
        t = time.time()
        try:
            arr, tr, how = fetch_year(y)
        except Exception as e:
            print(f"  {y}: unavailable ({e}); skipping"); continue
        NTL[y] = np.clip(np.nan_to_num(arr, nan=0.0), 0, None) * VALUE_SCALE
        NTL_TRANSFORM = tr
        print(f"  {y}: {arr.shape} via {how} in {time.time()-t:.0f}s | max {NTL[y].max():.1f} nW/cm2/sr")
        time.sleep(1)  # be polite to Zenodo
    if not NTL:
        raise RuntimeError("No night-lights year could be read. Check internet access (Kaggle: Settings > Internet on) and Zenodo status, then re-run.")
    np.savez_compressed(cache, transform=np.array(list(NTL_TRANSFORM)[:6]), **{f"y{y}": a for y, a in NTL.items()})

YEARS = sorted(NTL)
assert BASE_YEAR in YEARS and len(YEARS) >= 2, "need the base year plus at least one later year"
END_YEAR = YEARS[-1]
print(f"years {YEARS[0]}-{END_YEAR} | grid {NTL[END_YEAR].shape} | pixel {NTL_TRANSFORM.a*3600:.1f} arcsec")

## 3 · Method
1. **Rasterise** the 33 district polygons onto the night-lights grid (≈500 m pixels). Each pixel belongs to the district its centre falls in.
2. For every district and year, compute
   * **Sum of lights (SOL)**: total radiance, the standard "how much light" measure,
   * **mean radiance** = SOL ÷ pixels (comparable across big and small districts),
   * **lit share**: % of pixels ≥ `LIT_THRESHOLD`.
3. **Growth** = SOL(latest) ÷ SOL(2014) − 1, and **CAGR** = (SOL(latest)/SOL(2014))^(1/years) − 1. As a robustness check, the growth is recomputed on 2-year averages at each end.
4. **Hot-spots**: total light in a circle around approximate coordinates of known growth sites (ports, industrial estates, new cities).

In [ ]:
from rasterio.features import rasterize

H, W = NTL[END_YEAR].shape
ids = rasterize(((g, i + 1) for i, g in enumerate(districts.geometry)), out_shape=(H, W),
                transform=NTL_TRANSFORM, fill=0, dtype="int32")
n_pix = np.bincount(ids.ravel(), minlength=len(districts) + 1)[1:]

rows = []
for y in YEARS:
    a = NTL[y]
    sol = np.bincount(ids.ravel(), weights=a.ravel(), minlength=len(districts) + 1)[1:]
    lit = np.bincount(ids.ravel(), weights=(a >= LIT_THRESHOLD).ravel(), minlength=len(districts) + 1)[1:]
    for i, d in enumerate(districts.district):
        rows.append(dict(district=d, year=y, sol=sol[i], mean_rad=sol[i] / max(n_pix[i], 1),
                         lit_share=100 * lit[i] / max(n_pix[i], 1), pixels=int(n_pix[i])))
panel = pd.DataFrame(rows)
panel.to_csv(DATA_DIR / "gujarat_district_night_lights.csv", index=False)
SOL = panel.pivot(index="district", columns="year", values="sol")
MEAN = panel.pivot(index="district", columns="year", values="mean_rad")
print(panel.query("year == @END_YEAR").sort_values("sol", ascending=False).head(8).round(2).to_string(index=False))

In [ ]:
n_years = END_YEAR - BASE_YEAR
g = pd.DataFrame({"sol_base": SOL[BASE_YEAR], "sol_end": SOL[END_YEAR],
                  "mean_base": MEAN[BASE_YEAR], "mean_end": MEAN[END_YEAR]})
g["growth_pct"] = 100 * (g.sol_end / g.sol_base - 1)
g["cagr_pct"] = 100 * ((g.sol_end / g.sol_base) ** (1 / n_years) - 1)
g["abs_gain"] = g.sol_end - g.sol_base
g["base_share_pct"] = 100 * g.sol_base / g.sol_base.sum()
g["end_share_pct"] = 100 * g.sol_end / g.sol_end.sum()
# robustness: 2-year averages at both ends (dampens one noisy year)
b2 = [y for y in (BASE_YEAR, BASE_YEAR + 1) if y in SOL]; e2 = [y for y in (END_YEAR - 1, END_YEAR) if y in SOL]
g["growth_pct_2yavg"] = 100 * (SOL[e2].mean(1) / SOL[b2].mean(1) - 1)
g["eligible"] = g.base_share_pct >= 100 * MIN_BASE_SHARE
g = g.sort_values("growth_pct", ascending=False)
g["rank"] = np.arange(1, len(g) + 1)
growth = g.reset_index()

state = SOL.sum()
STATE_GROWTH = 100 * (state[END_YEAR] / state[BASE_YEAR] - 1)
STATE_CAGR = 100 * ((state[END_YEAR] / state[BASE_YEAR]) ** (1 / n_years) - 1)

# Version-break check: EOG changed VNL processing from V2.1 (to 2021) to V2.2 (2022+).
yoy = state.pct_change() * 100
print("Gujarat total lights, year-on-year %:\n", yoy.round(1).to_string())
if 2022 in yoy.index and 2021 in state.index:
    med = yoy.drop(index=2022, errors="ignore").median()
    print(f"\n2021->2022 change {yoy[2022]:+.1f}% vs median year {med:+.1f}%"
          + ("  <-- possible product-version step; read 2022+ levels with care" if abs(yoy[2022] - med) > 15 else "  (no obvious step)"))

cols = ["rank", "district", "sol_base", "sol_end", "growth_pct", "cagr_pct", "growth_pct_2yavg", "base_share_pct", "eligible"]
growth[cols].round(1)

In [ ]:
# ---- Hot-spots: light inside a circle around APPROXIMATE site coordinates (not official boundaries)
HOTSPOTS = [  # name, lat, lon, radius_km, district
    ("Mundra port area", 22.75, 69.70, 8, "Kachchh (Kutch)"),
    ("Khavda RE-park area", 23.84, 69.73, 20, "Kachchh (Kutch)"),
    ("Dahej industrial area", 21.70, 72.58, 8, "Bharuch"),
    ("GIFT City", 23.16, 72.68, 3, "Gandhinagar"),
    ("Hazira industrial area", 21.12, 72.65, 6, "Surat"),
    ("Dholera SIR", 22.25, 72.19, 10, "Ahmedabad"),
    ("Sanand industrial area", 22.99, 72.38, 6, "Ahmedabad"),
    ("Morbi ceramics cluster", 22.82, 70.84, 8, "Morbi"),
    ("Statue of Unity / Kevadia", 21.84, 73.72, 6, "Narmada"),
]
rr, cc = np.mgrid[0:H, 0:W]
lon_c = NTL_TRANSFORM.c + (cc + 0.5) * NTL_TRANSFORM.a
lat_c = NTL_TRANSFORM.f + (rr + 0.5) * NTL_TRANSFORM.e
hs_rows = []
for name, la, lo, rkm, dist in HOTSPOTS:
    dkm = np.hypot((lat_c - la) * 111.0, (lon_c - lo) * 111.0 * math.cos(math.radians(la)))
    m = dkm <= rkm
    s = {y: float(NTL[y][m].sum()) for y in YEARS}
    hs_rows.append(dict(site=name, district=dist, radius_km=rkm, sol_base=s[BASE_YEAR], sol_end=s[END_YEAR],
                        growth_pct=100 * (s[END_YEAR] / max(s[BASE_YEAR], 1e-6) - 1)))
del rr, cc, lon_c, lat_c
hotspots = pd.DataFrame(hs_rows).sort_values("growth_pct", ascending=False)
hotspots.round(1)

## 4 · Results: the headline number
The headline is computed here and never typed by hand. It names the district with the highest growth in total night lights since 2014
among districts that held at least `MIN_BASE_SHARE` of Gujarat's base-year light.

In [ ]:
elig = growth[growth.eligible]
top = elig.iloc[0]
HEAD = dict(district=top.district, growth_pct=float(top.growth_pct), cagr_pct=float(top.cagr_pct),
            base_year=BASE_YEAR, end_year=END_YEAR, state_growth=float(STATE_GROWTH), state_cagr=float(STATE_CAGR))
HEADLINE = (f"From space, {HEAD['district']} grew its lights {HEAD['growth_pct']:.0f}% since {BASE_YEAR} "
            f"— fastest in Gujarat.")
HOW = f"Sum of VIIRS night-time radiance inside district boundaries, {BASE_YEAR}→{END_YEAR} (Annual VNL V2, 500 m)."

unf = growth.iloc[0]
print(HEADLINE)
print(HOW)
print(f"Gujarat overall: {STATE_GROWTH:+.0f}% ({STATE_CAGR:.1f}%/yr).  Leader's CAGR: {HEAD['cagr_pct']:.1f}%/yr.")
print(f"Robustness (2-yr averages): {top.district} {top.growth_pct_2yavg:+.0f}%; "
      f"rank by 2-yr avg = {int(growth.growth_pct_2yavg.rank(ascending=False)[growth.district == top.district].iloc[0])}.")
if unf.district != top.district:
    print(f"Note: unfiltered leader is {unf.district} ({unf.growth_pct:+.0f}%) from a small base "
          f"({unf.base_share_pct:.2f}% of state light in {BASE_YEAR}).")
big = growth.sort_values("abs_gain", ascending=False).iloc[0]
print(f"Biggest absolute gain in light: {big.district} (+{big.abs_gain:,.0f} SOL units, {big.growth_pct:+.0f}%).")

In [ ]:
# Maps: base year vs latest (same scale), and growth
norm = LogNorm(vmin=max(0.05, np.nanpercentile(MEAN.values, 2)), vmax=np.nanpercentile(MEAN.values, 99.5))
gp = districts_plot.merge(growth[["district", "growth_pct"]], on="district")
fig, axs = plt.subplots(1, 3, figsize=(18, 6.4))
for ax, y in zip(axs[:2], (BASE_YEAR, END_YEAR)):
    districts_plot.assign(v=districts_plot.district.map(MEAN[y])).plot(
        column="v", cmap=NIGHT_CMAP, norm=norm, ax=ax, edgecolor=BG, linewidth=0.6)
    ax.set_title(f"Mean radiance {y}", color=FG); ax.set_axis_off()
sm = plt.cm.ScalarMappable(norm=norm, cmap=NIGHT_CMAP)
cb = fig.colorbar(sm, ax=axs[:2], shrink=0.6, pad=0.01); cb.set_label("nW/cm²/sr (log)", color=MUTED); tidy_log_colorbar(cb)
teal = LinearSegmentedColormap.from_list("teal", ["#13262A", "#1C6F69", TEAL, "#C9F4EF"])
gp.plot(column="growth_pct", cmap=teal, ax=axs[2], edgecolor=BG, linewidth=0.6, legend=True,
        legend_kwds={"shrink": 0.6, "label": "% growth in total light"})
axs[2].set_title(f"Growth {BASE_YEAR}→{END_YEAR}", color=FG); axs[2].set_axis_off()
for _, r in gp.nlargest(3, "growth_pct").iterrows():
    p = r.geometry.representative_point(); axs[2].annotate(r.district, (p.x, p.y), color=FG, fontsize=10, ha="center", path_effects=STROKE)
plt.savefig(FIG_DIR / "maps.png", dpi=110, bbox_inches="tight"); plt.show()

# Top-10 growth bars (single series -> one colour; labels in text ink)
t10 = growth.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(t10.district, t10.growth_pct, color=[ACCENT if e else MUTED for e in t10.eligible], height=0.62)
for i, (v, e) in enumerate(zip(t10.growth_pct, t10.eligible)):
    ax.text(v, i, f"  {v:+.0f}%" + ("" if e else "  (small base)"), va="center", color=FG, fontsize=10)
ax.axvline(STATE_GROWTH, color=TEAL, lw=1.5, ls="--"); ax.text(STATE_GROWTH, 9.6, f" Gujarat {STATE_GROWTH:+.0f}%", color=TEAL)
ax.set_xlabel(f"Growth in total night lights, {BASE_YEAR}→{END_YEAR} (%)"); ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.set_title("Fastest-growing districts by night lights", loc="left")
plt.savefig(FIG_DIR / "top10.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# Trajectories, indexed to 100 in the base year (one axis, comparable shapes)
SHOW = list(dict.fromkeys([HEAD["district"], "Kachchh (Kutch)", "Bharuch", "Gandhinagar", "Surat", "Ahmedabad"]))
SHOW = [d for d in SHOW if d in SOL.index][:6]
idx = SOL.loc[SHOW].div(SOL.loc[SHOW, BASE_YEAR], axis=0) * 100
st_idx = state / state[BASE_YEAR] * 100
fig, ax = plt.subplots(figsize=(10, 5.5))
pal = [ACCENT, TEAL, "#7AA2F7", "#E0AF68", "#BB9AF7", "#9ECE6A"]
for (d, row), c in zip(idx.iterrows(), pal):
    ax.plot(row.index, row.values, color=c, lw=2.4 if d == HEAD["district"] else 2, label=d)
    ax.text(row.index[-1] + 0.15, row.values[-1], d, color=FG, fontsize=9, va="center")
ax.plot(st_idx.index, st_idx.values, color=MUTED, lw=2, ls="--", label="Gujarat total")
ax.set_ylabel(f"Total night lights (index, {BASE_YEAR} = 100)"); ax.grid(color=GRID)
ax.set_xlim(YEARS[0] - 0.3, END_YEAR + 2.2); ax.set_xticks(YEARS[::2]); ax.legend(frameon=False, loc="upper left", fontsize=9)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.set_title("District trajectories", loc="left")
plt.savefig(FIG_DIR / "trajectories.png", dpi=110, bbox_inches="tight"); plt.show()

### Lights vs. a published economic indicator: skipped
We looked for an **open, machine-readable district-level GDP (GDDP) table for Gujarat** and did not find one. District
Domestic Product estimates exist only inside PDF reports, with mixed base years. So we **do not fit a lights→GDP
regression** and make no rupee claims. Light growth here is a *proxy* for activity, not a measured GDP number.
If you have a CSV with columns `district, year, gddp`, set `GDDP_CSV` below and the cell fits a simple log-log line.

In [ ]:
GDDP_CSV = None   # e.g. "my_gujarat_gddp.csv" (district, year, gddp). Left empty: no open table found.

def fit_lights_vs_gddp(panel, gddp):
    m = panel.merge(gddp, on=["district", "year"]).query("sol > 0 and gddp > 0")
    x, y = np.log(m.sol.values), np.log(m.gddp.values)
    b, a = np.polyfit(x, y, 1); r2 = np.corrcoef(x, y)[0, 1] ** 2
    print(f"log(GDDP) = {a:.2f} + {b:.2f}·log(SOL)   R² = {r2:.2f}   n = {len(m)}")
    return b, a, r2

if GDDP_CSV:
    fit_lights_vs_gddp(panel, pd.read_csv(GDDP_CSV))
else:
    print("Regression skipped: no open district GDP table for Gujarat (see markdown above).")

## 5 · Interactive demo: ask the district table
Type a question such as *"How did Kutch do?"*, *"Compare Surat and Bharuch"*, *"Which districts grew fastest?"* or *"Hot-spots?"*.
Every answer comes from a **deterministic lookup** over the computed table. If a GPU is available and `USE_LLM=True`, a small
open model (Qwen2.5-1.5B-Instruct) rewrites those looked-up facts in plain English and gets no other information.
The demo works with or without the GPU.

In [ ]:
import re
ALIASES = {d.lower(): d for d in growth.district}
ALIASES.update({"kutch": "Kachchh (Kutch)", "kachchh": "Kachchh (Kutch)", "ahmadabad": "Ahmedabad", "dohad": "Dahod",
    "mahesana": "Mehsana", "banas kantha": "Banaskantha", "sabar kantha": "Sabarkantha", "panch mahals": "Panchmahal",
    "panchmahals": "Panchmahal", "dangs": "Dang", "the dangs": "Dang", "chota udaipur": "Chhota Udaipur",
    "devbhoomi dwarka": "Devbhumi Dwarka", "dwarka": "Devbhumi Dwarka", "baroda": "Vadodara", "morvi": "Morbi",
    "gift city": "Gandhinagar", "mundra": "Kachchh (Kutch)", "khavda": "Kachchh (Kutch)", "dahej": "Bharuch",
    "hazira": "Surat", "dholera": "Ahmedabad", "sanand": "Ahmedabad", "kevadia": "Narmada"})
ALIASES = {k: v for k, v in ALIASES.items() if v in set(growth.district)}

def find_districts(q):
    ql = q.lower(); hits = []
    for k in sorted(ALIASES, key=len, reverse=True):
        if re.search(r"\b" + re.escape(k) + r"\b", ql) and ALIASES[k] not in hits:
            hits.append(ALIASES[k]); ql = ql.replace(k, " ")
    return hits

def profile(d):
    r = growth.set_index("district").loc[d]
    return (f"{d}: total lights {r.sol_base:,.0f} → {r.sol_end:,.0f} ({BASE_YEAR}→{END_YEAR}), "
            f"{r.growth_pct:+.0f}% ({r.cagr_pct:.1f}%/yr), rank {int(r['rank'])} of {len(growth)}; "
            f"mean radiance {r.mean_end:.2f} nW/cm²/sr; {r.end_share_pct:.1f}% of Gujarat's light.")

def answer_lookup(q):
    ql = q.lower(); ds = find_districts(q)
    if any(w in ql for w in ("hot", "site", "port", "hotspot", "hot-spot")) and not ds:
        return "Hot-spots (approx. circles):\n" + "\n".join(
            f"• {r.site} ({r.district}): {r.growth_pct:+.0f}%" for r in hotspots.itertuples())
    if len(ds) >= 2:
        return "\n".join(profile(d) for d in ds[:4])
    if ds:
        extra = [f"  • {r.site}: {r.growth_pct:+.0f}%" for r in hotspots.itertuples() if r.district == ds[0]]
        return profile(ds[0]) + (("\nSites inside:\n" + "\n".join(extra)) if extra else "")
    if any(w in ql for w in ("slow", "least", "lowest", "worst", "bottom", "declin")):
        b = growth.tail(5)
        return "Slowest growth:\n" + "\n".join(f"• {r.district}: {r.growth_pct:+.0f}%" for r in b.itertuples())
    if any(w in ql for w in ("bright", "most light", "largest", "biggest")):
        b = growth.sort_values("sol_end", ascending=False).head(5)
        return f"Brightest in {END_YEAR} (total light):\n" + "\n".join(
            f"• {r.district}: {r.end_share_pct:.1f}% of Gujarat" for r in b.itertuples())
    if any(w in ql for w in ("gujarat", "state", "overall", "total")) and not any(w in ql for w in ("fast", "top", "grew")):
        return f"Gujarat overall: {STATE_GROWTH:+.0f}% total night lights {BASE_YEAR}→{END_YEAR} ({STATE_CAGR:.1f}%/yr)."
    if any(w in ql for w in ("fast", "top", "most", "grew", "growth", "best", "rank", "highest")):
        b = growth.head(5)
        return HEADLINE + "\nTop 5 by growth:\n" + "\n".join(
            f"• {r.district}: {r.growth_pct:+.0f}%" + ("" if r.eligible else " (small base)") for r in b.itertuples())
    return ("I can answer about any Gujarat district (e.g. 'How did Kutch do?'), comparisons ('Surat vs Bharuch'), "
            "'fastest', 'slowest', 'brightest', 'Gujarat overall', or 'hot-spots'.")

# ---- optional small LLM (GPU only); it may only rephrase looked-up facts
LLM = None
try:
    import torch
    if USE_LLM and torch.cuda.is_available():
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        mdl = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="cuda")
        LLM = (tok, mdl); print("LLM loaded:", LLM_MODEL)
    else:
        print("No GPU (or USE_LLM=False): deterministic answers only.")
except Exception as e:
    print("LLM unavailable, deterministic answers only:", str(e)[:120])

def answer(q):
    facts = answer_lookup(q)
    if LLM is None:
        return facts
    tok, mdl = LLM
    msgs = [{"role": "system", "content": "You answer questions about Gujarat district night-light growth. Use ONLY the "
             "FACTS given. Do not add numbers. Say lights are a proxy for economic activity, not GDP. Max 80 words."},
            {"role": "user", "content": f"QUESTION: {q}\nFACTS:\n{facts}"}]
    ids_ = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(mdl.device)
    out = mdl.generate(ids_, max_new_tokens=160, do_sample=False)
    text = tok.decode(out[0, ids_.shape[1]:], skip_special_tokens=True).strip()
    return f"{text}\n\n— looked-up facts —\n{facts}"

for q in ["How did Kutch do?", "Compare Surat and Bharuch", "Which districts grew fastest?"]:
    print(">>", q, "\n" + answer_lookup(q), "\n")

In [ ]:
import gradio as gr
demo = gr.Interface(fn=answer, inputs=gr.Textbox(label="Ask about Gujarat's night lights", lines=2),
                    outputs=gr.Textbox(label="Answer", lines=10),
                    examples=["How did Kutch do?", "Compare Surat and Bharuch", "Which districts grew fastest?",
                              "Hot-spots?", "Gujarat overall"],
                    title="23N Field Test #03 · Gujarat night lights Q&A",
                    description="Answers come from the district table computed above (VIIRS Annual VNL V2).")
if os.environ.get("FT_NO_LAUNCH") is None:
    try:
        demo.launch(share=True, prevent_thread_lock=True)
    except Exception as e:
        print("Gradio launch skipped:", e)

## 6 · Video render (1080×1080, 30 fps, about 58 s, H.264) and cover (1080×1350)
Hook (0–5 s) → animated district map with year counter (5–27 s) → top-5 growth bars (27–40 s) → headline (40–52 s) → end card (52–58 s).
Captions are burned in because most people watch muted.

In [ ]:
def _wrap_text(s, n):
    import textwrap
    return "\n".join(textwrap.wrap(s, n))

def render_video(gdf_plot, years, values, top5, head, out="ft03_video.mp4", fps=30,
                 seg=(5, 22, 13, 12, 6), field_test="03"):
    """gdf_plot: GeoDataFrame with 'district' + geometry; values: DataFrame district x year (mean radiance);
    top5: DataFrame [district, growth_pct]; head: dict(district, growth_pct, base_year, end_year, how)."""
    W = H = 1080; dpi = 100
    proc = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
                             "-s", f"{W}x{H}", "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "veryfast",
                             "-pix_fmt", "yuv420p", "-movflags", "+faststart", out], stdin=subprocess.PIPE)
    fig = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi, facecolor=BG)
    def grab():
        fig.canvas.draw(); return np.asarray(fig.canvas.buffer_rgba())[..., :3].tobytes()
    def caption(t, y=0.055, size=24, color=FG):
        fig.text(0.5, y, _wrap_text(t, 60), ha="center", va="center", fontsize=size, color=color,
                 bbox=dict(boxstyle="round,pad=0.5", fc="#000000", ec="none", alpha=0.55))
    def brand():
        fig.text(0.04, 0.965, "23N FIELD TESTS · #" + field_test, fontsize=15, color=ACCENT, weight="bold", va="top")
    def write_static(n, fade=12):
        frame = np.frombuffer(grab(), np.uint8).reshape(H, W, 3)
        bg = np.array(matplotlib.colors.to_rgb(BG)) * 255
        for k in range(n):
            a = min(1.0, (k + 1) / fade) if fade else 1.0
            proc.stdin.write((frame * a + bg * (1 - a)).astype(np.uint8).tobytes() if a < 1 else frame.tobytes())
    n_hook, n_map, n_bar, n_head, n_end = [int(s * fps) for s in seg]

    # 1) hook
    fig.clf(); brand()
    fig.text(0.5, 0.56, "Can you measure\nGujarat's growth\nfrom space?", ha="center", va="center",
             fontsize=64, weight="bold", color=FG, linespacing=1.15)
    fig.text(0.5, 0.30, f"VIIRS night-time lights · {len(gdf_plot)} districts · {years[0]}–{years[-1]}", ha="center",
             fontsize=24, color=TEAL)
    write_static(n_hook)

    # 2) animated choropleth with year counter (interpolated between years)
    fig.clf(); brand()
    ax = fig.add_axes([0.03, 0.11, 0.94, 0.70]); ax.set_axis_off()
    vals = values.loc[gdf_plot.district, years].to_numpy(dtype=float)
    norm_ = LogNorm(vmin=max(0.05, np.nanpercentile(vals, 2)), vmax=max(np.nanpercentile(vals, 99.5), 0.1))
    gdf_plot.plot(ax=ax, color="#141B24", edgecolor=BG, linewidth=0.8)
    coll = ax.collections[-1]
    hi = gdf_plot[gdf_plot.district == head["district"]]
    fig.text(0.04, 0.905, "Night-time brightness by district", fontsize=30, weight="bold", color=FG)
    ytxt = fig.text(0.95, 0.80, "", ha="right", fontsize=72, weight="bold", color=ACCENT)
    caption("Each district coloured by its average night-time brightness (VIIRS, 500 m)")
    cax = fig.add_axes([0.06, 0.17, 0.30, 0.018])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm_, cmap=NIGHT_CMAP), cax=cax, orientation="horizontal")
    cb.ax.tick_params(colors=MUTED, labelsize=13); cb.outline.set_visible(False); tidy_log_colorbar(cb, horizontal=True)
    cax.set_title("nW/cm²/sr", color=MUTED, fontsize=12, loc="left")
    hold = int(2.5 * fps); n_anim = n_map - hold; outline = None
    for k in range(n_map):
        t = min(k / max(n_anim - 1, 1), 1.0) * (len(years) - 1)
        i0 = int(np.floor(t)); i1 = min(i0 + 1, len(years) - 1); w = t - i0
        v = np.exp((1 - w) * np.log(np.clip(vals[:, i0], 1e-3, None)) + w * np.log(np.clip(vals[:, i1], 1e-3, None)))
        coll.set_facecolor(NIGHT_CMAP(norm_(v)))
        ytxt.set_text(str(years[int(round(t))]))
        if k == n_anim and len(hi):
            hi.boundary.plot(ax=ax, color=TEAL, linewidth=3)
            p = hi.geometry.iloc[0].representative_point()
            ax.annotate(head["district"], (p.x, p.y), color=FG, fontsize=22, weight="bold", ha="center",
                        bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, alpha=0.85))
        proc.stdin.write(grab())

    # 3) top-5 bar chart, bars grow in
    fig.clf(); brand()
    fig.text(0.04, 0.905, "Fastest-growing lights", fontsize=34, weight="bold", color=FG)
    fig.text(0.04, 0.865, f"Growth in total night-time light, {head['base_year']}→{head['end_year']}",
             fontsize=20, color=MUTED)
    ax = fig.add_axes([0.30, 0.18, 0.60, 0.62])
    t5 = top5.iloc[::-1].reset_index(drop=True); vmax = float(t5.growth_pct.max()) * 1.25
    bars = ax.barh(t5.district, [0] * len(t5), color=[ACCENT if i == len(t5) - 1 else TEAL for i in range(len(t5))],
                   height=0.6)
    ax.set_xlim(min(0, float(t5.growth_pct.min()) * 1.25), vmax if vmax > 0 else 1); ax.tick_params(axis="y", labelsize=22, colors=FG, length=0)
    ax.tick_params(axis="x", labelsize=14); ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
    for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
    labels = [ax.text(0, i, "", va="center", fontsize=22, color=FG, weight="bold") for i in range(len(t5))]
    caption("Total light = sum of satellite radiance inside each district's boundary")
    grow = int(2.5 * fps)
    for k in range(n_bar):
        f = min(1.0, (k + 1) / grow); f = 1 - (1 - f) ** 3
        for b, lab, v, i in zip(bars, labels, t5.growth_pct, range(len(t5))):
            b.set_width(v * f); lab.set_x(v * f); lab.set_text(f"  {v * f:+.0f}%")
        proc.stdin.write(grab())

    # 4) headline + mini map
    fig.clf(); brand()
    fig.text(0.5, 0.80, f"{head['growth_pct']:+.0f}%", ha="center", fontsize=150, weight="bold", color=ACCENT)
    fig.text(0.5, 0.70, f"{head['district']}", ha="center", fontsize=46, weight="bold", color=FG)
    fig.text(0.5, 0.645, f"fastest-growing night lights in Gujarat, {head['base_year']}→{head['end_year']}",
             ha="center", fontsize=24, color=FG)
    ax = fig.add_axes([0.22, 0.17, 0.56, 0.44]); ax.set_axis_off()
    gdf_plot.plot(ax=ax, color="#1B2430", edgecolor=BG, linewidth=0.6)
    if len(hi): hi.plot(ax=ax, color=ACCENT, edgecolor=FG, linewidth=1)
    caption("How: " + head["how"], size=19, color=MUTED)
    write_static(n_head)

    # 5) end card
    fig.clf()
    fig.text(0.5, 0.58, "23N AI Labs", ha="center", fontsize=72, weight="bold", color=FG)
    fig.text(0.5, 0.49, f"Field Test #{field_test} · Open notebook in comments", ha="center", fontsize=30, color=ACCENT)
    fig.text(0.5, 0.40, "23nlabs.com", ha="center", fontsize=34, color=TEAL)
    fig.text(0.5, 0.10, "Independent study on public data · VIIRS Annual VNL V2 (CC BY 4.0)", ha="center",
             fontsize=16, color=MUTED)
    write_static(n_end)

    proc.stdin.close(); rc = proc.wait(); plt.close(fig)
    if rc != 0: raise RuntimeError("ffmpeg failed")
    return out

def render_cover(gdf_plot, values, year, head, out="ft03_cover.png"):
    fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor=BG)
    fig.text(0.05, 0.955, "23N FIELD TESTS · #03", fontsize=18, color=ACCENT, weight="bold", va="top")
    fig.text(0.05, 0.915, "Gujarat from Space", fontsize=50, weight="bold", color=FG, va="top")
    fig.text(0.05, 0.855, "Night lights as an economic pulse", fontsize=26, color=MUTED, va="top")
    ax = fig.add_axes([0.03, 0.25, 0.94, 0.56]); ax.set_axis_off()
    v = gdf_plot.district.map(values[year])
    n_ = LogNorm(vmin=max(0.05, np.nanpercentile(values.values, 2)), vmax=max(np.nanpercentile(values.values, 99.5), 0.1))
    gdf_plot.assign(v=v).plot(column="v", cmap=NIGHT_CMAP, norm=n_, ax=ax, edgecolor=BG, linewidth=0.7)
    hi = gdf_plot[gdf_plot.district == head["district"]]
    if len(hi): hi.boundary.plot(ax=ax, color=TEAL, linewidth=3)
    fig.text(0.05, 0.19, f"{head['growth_pct']:+.0f}%", fontsize=96, weight="bold", color=ACCENT)
    fig.text(0.05, 0.12, f"{head['district']}: fastest-growing lights, {head['base_year']}→{head['end_year']}",
             fontsize=26, color=FG)
    fig.text(0.05, 0.05, "23nlabs.com · open notebook · VIIRS Annual VNL V2", fontsize=18, color=TEAL)
    fig.savefig(out, dpi=100, facecolor=BG); plt.close(fig)
    return out

In [ ]:
t = time.time()
top5 = growth[growth.eligible].head(5)[["district", "growth_pct"]]
HEAD["how"] = HOW
video = render_video(districts_plot, YEARS, MEAN, top5, HEAD, out="ft03_video.mp4")
cover = render_cover(districts_plot, MEAN, END_YEAR, HEAD, out="ft03_cover.png")
probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                        "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", video],
                       capture_output=True, text=True)
print(json.loads(probe.stdout), f"| rendered in {time.time()-t:.0f}s")
try:
    from google.colab import files
    files.download(video); files.download(cover)
except Exception:
    print("Not on Colab: files saved next to the notebook:", video, cover)

## 7 · Post text (filled with the computed numbers, ready to paste)

In [ ]:
LINKEDIN_TEMPLATE = """Can you measure Gujarat's growth from space?

{headline}

What we did: summed VIIRS satellite night-time light inside each of Gujarat's {n_districts} district boundaries, every year {base_year}→{end_year}.

What stood out:
• Gujarat overall: {state_growth} total light ({state_cagr}/yr)
• Top 3 by growth: {top3}
• Biggest absolute gain: {abs_leader}
• Fastest hot-spot: {hotspot_line}

Night lights are a proxy, not GDP. They pick up electrification, industry, ports and new cities, and they miss services and anything that doesn't glow. No open district-GDP table exists for Gujarat, so we did not fit one.

The notebook is open. Run it in free Colab, change the base year, and ask the table questions in plain English.

Data: Annual VNL V2 (EOG) via OpenGeoHub/Zenodo, CC BY 4.0. Independent study on public data.

23N Field Tests #03 · 23N AI Labs, Gandhinagar
Notebook link in comments.
#Gujarat #RemoteSensing #NightLights #OpenData #AI"""

X_TEMPLATE = """Can you measure Gujarat's growth from space?

{headline_short}

VIIRS night lights summed over {n_districts} districts, {base_year}→{end_year}. Gujarat overall {state_growth}.

Open notebook 👇 #23NFieldTests"""

X_THREAD = ["1/ {headline_short} Method: sum of VIIRS night-time radiance inside each district, {base_year}→{end_year}.",
            "2/ Top 3 by growth: {top3}. Fastest hot-spot: {hotspot_line}.",
            "3/ Lights ≠ GDP. They miss services and anything that doesn't glow. Open notebook: {notebook_url}"]

t3 = growth[growth.eligible].head(3)
hs = hotspots.iloc[0]
vals = dict(
    headline=HEADLINE,
    headline_short=f"From space, {HEAD['district']} grew its lights {HEAD['growth_pct']:.0f}% since {BASE_YEAR}, fastest in Gujarat.",
    base_year=BASE_YEAR, end_year=END_YEAR,
    state_growth=f"{STATE_GROWTH:+.0f}%", state_cagr=f"{STATE_CAGR:.1f}%",
    top3=", ".join(f"{r.district} {r.growth_pct:+.0f}%" for r in t3.itertuples()),
    abs_leader=f"{big.district} ({big.end_share_pct:.0f}% of the state's light in {END_YEAR})",
    hotspot_line=f"{hs.site} {hs.growth_pct:+.0f}% (approx. {hs.radius_km} km circle)",
    n_districts=len(growth), notebook_url=NOTEBOOK_URL)
li = LINKEDIN_TEMPLATE.format(**vals); x = X_TEMPLATE.format(**vals)
print(li, f"\n\n[{len(li)} chars; LinkedIn limit 1,300 for this series]\n" + "-" * 60)
print(x, f"\n\n[{len(x)} chars; X limit 280]\n" + "-" * 60)
for tw in X_THREAD: s = tw.format(**vals); print(s, f"  [{len(s)}]")
if len(li) > 1300: print("WARNING: LinkedIn text over 1,300 chars; trim the bullets.")
if len(x) > 280: print("WARNING: X text over 280 chars.")
print(f"\nTotal notebook runtime so far: {(time.time()-T0)/60:.1f} min")

## 8 · Limitations (read before quoting the number)
* **Lights ≠ GDP.** Night lights track electrification, industry, ports, highways and urban spread. They under-count services, IT, finance and agriculture, and over-weight a few very bright industrial sites. A district can grow economically without glowing more.
* **Sensor-product step.** EOG moved Annual VNL from V2.1 (2012–2021) to V2.2 (2022+), which changed the background masking. The notebook prints the 2021→2022 state-level jump. If it looks like a step, compare districts with each other rather than reading absolute levels, and use the 2-year-average growth column.
* **Source processing.** We use OpenGeoHub's repackaging (values stored ×10, capped at 200 nW/cm²/sr), not EOG's original files. Saturated city cores (above the cap) can understate growth in the brightest districts such as Ahmedabad and Surat. We did not separately remove gas flares or brightly lit industrial plants.
* **Small bases inflate percentages.** A dark district that adds a few lit villages can post a huge %. The headline therefore requires ≥ `MIN_BASE_SHARE` of base-year state light. The unfiltered leader and the biggest absolute gainer are printed too.
* **Boundaries.** The 33-district map (2014-era LGD) has no Vav-Tharad, which was carved out of Banaskantha in 2025. All years use the same boundaries. Pixels on a border go to the district holding the pixel centre (~500 m). Boundary file: community-made (MIT repo), not an official survey map.
* **Hot-spots** are circles around *approximate* coordinates, not official site boundaries. Treat them as indicative.
* **Clouds, monsoon and moonlight** are handled by EOG's annual compositing, but residual noise remains. Treat single-year blips with care.
* **No regression to GDP.** We found no open, machine-readable district GDP table for Gujarat, so we do not translate lights into rupees.
* Independent study on public data. Not affiliated with, or endorsed by, any company, government agency or data provider named here.